In [7]:
"""
Compute average training times for various cross-validation runs from full training info
and add it to table 'MLP training details'.
"""
# pylint: disable=redefined-outer-name
import json
import lzma
import re
from datetime import timedelta
from pathlib import Path

import pandas as pd
from IPython.display import display

from epiclass.utils.notebooks.paper.paper_utilities import ASSAY

In [2]:
table_file = (
    Path.home()
    / "downloads"
    / "EpiClass Supplementary tables - ST17-MLP training details.csv"
)
training_details_df = pd.read_csv(table_file, skiprows=1)
training_details_df.head()

,Dataset,feature_set_name,metadata_category,split,Name,output_size,Total nb of files,train size,validation size,tra_Accuracy,tra_F1Score,val_Accuracy,val_F1Score,Experiment key,Code version / commit,Exact commit,"Server start time (Unix, ms)",Date (YYYY-MM-DD),comet-url
0,2023-01-epiatlas-freeze,hg38_100kb_all_none,assay_epiclass,split0,hg38_100kb_all_none-assay_epiclass_1l_3000n-10...,11,21606,38487,Unknown,0.9983,0.9978,0.9935,0.9874,f56c92e557b340a8979341a64b59cbb5,v0.3.4,66a0a0b1882d3b958361611dd53538c8595d6471,1674861112548,2023-01-27,https://www.comet.com/rabyj/epiclass/f56c92e55...
1,2023-01-epiatlas-freeze,hg38_100kb_all_none,assay_epiclass,split1,hg38_100kb_all_none-assay_epiclass_1l_3000n-10...,11,21606,38389,Unknown,0.9942,0.9924,0.9912,0.9842,c7296f23a54142ac979e12055265add6,v0.3.4,66a0a0b1882d3b958361611dd53538c8595d6471,1674862555274,2023-01-27,https://www.comet.com/rabyj/epiclass/c7296f23a...
2,2023-01-epiatlas-freeze,hg38_100kb_all_none,assay_epiclass,split2,hg38_100kb_all_none-assay_epiclass_1l_3000n-10...,11,21606,38411,Unknown,0.9991,0.9988,0.9908,0.9859,2f3f7d19085d41a8a9b5a89b561e3592,v0.3.4,66a0a0b1882d3b958361611dd53538c8595d6471,1674863352444,2023-01-27,https://www.comet.com/rabyj/epiclass/2f3f7d190...
3,2023-01-epiatlas-freeze,hg38_100kb_all_none,assay_epiclass,split3,hg38_100kb_all_none-assay_epiclass_1l_3000n-10...,11,21606,38335,Unknown,0.9991,0.9986,0.9949,0.9917,edcdd6caebc343618a8383da7b43264f,v0.3.4,66a0a0b1882d3b958361611dd53538c8595d6471,1674864700151,2023-01-28,https://www.comet.com/rabyj/epiclass/edcdd6cae...
4,2023-01-epiatlas-freeze,hg38_100kb_all_none,assay_epiclass,split4,hg38_100kb_all_none-assay_epiclass_1l_3000n-10...,11,21606,38495,Unknown,0.9989,0.9983,0.9945,0.9909,22853835392146ec9bf90963a5179201,v0.3.4,66a0a0b1882d3b958361611dd53538c8595d6471,1674866124703,2023-01-28,https://www.comet.com/rabyj/epiclass/228538353...


In [3]:
display(training_details_df["Dataset"].value_counts())

Dataset
dfreeze_v2                 454
2023-01-epiatlas-freeze     60
Name: count, dtype: int64

In [4]:
training_details_df["oversampling"] = training_details_df["Name"].str.contains(
    "oversampl", case=False, regex=False
)
training_details_df["cross-validation"] = training_details_df["Name"].str.contains(
    "fold", case=False, regex=False
)

In [9]:
full_details_file = (
    Path.home()
    / "Projects/epiclass/output/logs/comet_ml_all_experiments_full_2026-02-04.json.xz"
)

with lzma.open(full_details_file, "rt", encoding="utf-8") as f:
    full_details = json.load(f)

In [10]:
full_details = {
    k: v
    for k, v in full_details.items()
    if k in training_details_df["Experiment key"].values
}

In [11]:
def parse_time_to_seconds(t):
    """Convert different time formats to seconds."""

    # Case 1: already timedelta object
    if isinstance(t, timedelta):
        return t.total_seconds()

    # Case 2: string like "0:36:35"
    if isinstance(t, str) and ":" in t:
        h, m, s = map(int, t.split(":"))
        return h * 3600 + m * 60 + s

    # Case 3: string like "datetime.timedelta(seconds=8108)"
    if isinstance(t, str) and "timedelta" in t:
        match = re.search(r"seconds=(\d+)", t)
        if match:
            return int(match.group(1))

    raise ValueError(f"Unknown time format: {t}")


def format_seconds_to_hms(seconds):
    """Format seconds into H:M:S format."""
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h}:{m:02d}:{s:02d}"

In [12]:
# Save one example of the full details for easier inspection
first_exp = list(full_details.keys())[0]
output_path = Path.home() / "downloads" / "sample_full_details.json"
# with open(output_path, "w", encoding="utf8") as f:
#     json.dump({first_exp: full_details[first_exp]}, f, indent=4)

In [13]:
times_sec = {}
input_sizes = {}

for exp_key in training_details_df["Experiment key"]:
    data = full_details[exp_key]
    metrics = data["metrics"]
    params_summary = data["parameters_summary"]

    for metric_dict in metrics:
        metric_name = metric_dict["metricName"]
        if "Training" in metric_name:
            metric_value = metric_dict["metricValue"]
            seconds = parse_time_to_seconds(metric_value)
            times_sec[exp_key] = seconds
            break

    for param_dict in params_summary:
        param_name = param_dict["name"]
        if param_name == "input_size":
            param_value = param_dict["valueCurrent"]
            input_sizes[exp_key] = int(param_value)
            break

    if exp_key not in times_sec:
        print(f"Warning: No training time found for experiment {exp_key}")
        times_sec[exp_key] = None

    if exp_key not in input_sizes:
        print(f"Warning: No input size found for experiment {exp_key}")
        input_sizes[exp_key] = None

In [14]:
training_details_df.loc[:, "training_time_sec"] = training_details_df[
    "Experiment key"
].map(times_sec)
training_details_df.loc[:, "input_size"] = training_details_df["Experiment key"].map(
    input_sizes
)

In [15]:
training_details_df["training time (minutes)"] = round(
    training_details_df["training_time_sec"] / 60
)

In [16]:
new_table_path = table_file.stem + "_with_times.csv"
# training_details_df.to_csv(new_table_path, index=False)

In [17]:
grouping_columns = [
    "Dataset",
    "feature_set_name",
    "metadata_category",
    "input_size",
    "output_size",
    "Total nb of files",
    "oversampling",
]

In [23]:
col = training_details_df["cross-validation"]
assert col.dtype == bool
assert col.notna().all()
sub_df = training_details_df[col]

In [ ]:
filter1 = sub_df["metadata_category"] == ASSAY
filter2 = sub_df["output_size"] == 7
filter3 = sub_df["Dataset"] == "dfreeze_v2"
test = sub_df[filter1 & filter2 & filter3]

In [ ]:
display(test)

In [ ]:
# Group by and compute mean & std
grouped = (
    sub_df.groupby(grouping_columns)["training_time_sec"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

# Convert to minutes
grouped["mean_min"] = (grouped["mean"] / 60).round(1)
grouped["std_min"] = (grouped["std"] / 60).round(1)
grouped["count"] = grouped["count"].astype(int)

# Combine into a single string
grouped["training_time (min)"] = (
    grouped["mean_min"].astype(str) + " ± " + grouped["std_min"].astype(str)
)

# Keep only relevant columns
grouped_summary = grouped[grouping_columns + ["training_time (min)", "count"]]
display(grouped_summary)